### **PROJETO FINAL - ANALISE EXPORATORIA DE DADOS - JUNHO 2026**

**Diagnóstico da carteira de cartões - NeoCard**

A empresa NeoCard encaminhou base de dados contendo a carteira ativa de clientes de diferentes segmentos e regiões, contendo relatório de consumo via cartão. Para a elaboração de um diagnóstico analítico é necessário:
- Entender quem são os clientes por segmentos e regiões, comportamento do consumo via cartão;
- Detectar a qualidade dos dados;
- Apresentar um relatório direto, claro visualmente com conclusões para tomada de decisão.

> **1º - Importando a biblioteca PANDAS (manipulação e análise de dados), SEABORN , MATPLOTLIB (visualição de dados)**

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

>**2º Importando o dataset neocard_dataset. Trata-se de arquivo excel (xlsx), que contem duas planilhas chamadas clientes e transacoes**

In [ ]:
clientes = pd.read_excel("neocard_dataset.xlsx", sheet_name="clientes")
transacoes  = pd.read_excel("neocard_dataset.xlsx", sheet_name="transacoes")

>**3º Análise dataset**

- <u>Planilha clientes</u>

In [ ]:
clientes.shape #identificando tamanho da tabela

In [ ]:
clientes.head() #primeiras linhas planilha clientes

In [ ]:
clientes.info() #tipo de dados por colunas, e qtd de valores não nulos

In [ ]:
clientes['id_cliente'].duplicated().sum() #verificando se os valores estão duplicados (chave do dataset)

In [ ]:
clientes.isna().sum() #qtd de valores nulos por coluna

In [ ]:
clientes[["regiao", "segmento_cartao", "canal_aquisicao", "produto_core"]].apply(lambda x: x.unique()) #valores unicos

In [ ]:
clientes[["idade", "score_credito", "limite_credito", "tempo_relacionamento_meses"]].describe().round(2) #metricas principais

A planilha clientes possui 9 colunas e 4000 registros, contendo o id do cliente, idade, região, segmento, score, limite do cartão, tempo de relacionamento, canal de aquisição do cartão e o tipo do produto utilizado.
Identificação das colunas:
- nºs inteiros(int64):
    - id_cliente -> não possui chaves duplicadas (chave primaria)
    - idade -> realizar modificações, possui valores fora do parâmetro normal para "idade"
    - tempo_relacionamento_meses -> valores em meses
- nºs decimais(float):
    - score_credito -> analisar alterações, possui 180 registros nulos
    - limite_credito -> valores bm distintos, verificar outliers
- texto(object):
    - regiao -> possui 5 valores distintos
    - segmento_cartao -> possui 4 valores distintos
    - canal_aquisicao -> possui 4 valores distintos
    - produto_core -> apenas um valor para todos os clientes


- <u>Planilha transacoes</u>

In [ ]:
transacoes.shape

In [ ]:
transacoes.info()

In [ ]:
transacoes.head()

In [ ]:
transacoes.isna().sum()

In [ ]:
transacoes['id_transacao'].duplicated().sum()

In [ ]:
transacoes[["categoria", "canal_transacao"]].apply(lambda x: x.unique())

In [ ]:
sorted(transacoes['categoria'].unique().tolist()) #utilizado pois no unique não trouxe todos os valores de categoria

In [ ]:
print(len(sorted(transacoes['categoria'].unique().tolist())))
print()
print(transacoes['categoria'].value_counts(dropna=False))

In [ ]:
transacoes[["data_transacao", "valor_transacao"]].describe().round(2)

A planilha transacoes possui 6 colunas e 216.061 registros, contendo o id da transação, id do cliente, data da transação, local da compra, valor da compra, canal utilizado para efetuar a compra.

Identificação das colunas:
- nºs inteiros(int64):
    - id_transacao -> não possui chaves duplicadas (chave primaria)
    - id_cliente -> chave estrangeira
- nºs decimais(float):
    - valor_transacao -> analisar. Contem valores negativos e 2161 valores nulos
- data(datetime64):
    - data_transacao -> periodo de 01/01/2025 a 30/12/2025
- texto(object):
    - categoria -> possui 20 valores, regularizar pois estão duplicados
    - canal_transacao -> possui 2 valores distintos

>**4º Tratamento dos dados**

- <u>Verificando a possibilidade de exclusão dos clientes com idade incoerente</u>

In [ ]:
clientes[(clientes["idade"] >=90) | (clientes["idade"] < 18)][['id_cliente', 'idade']] #trazendo os clientes com idade maior do que 90 anos e menor do que 18

In [ ]:
print(len(clientes[(clientes["idade"] >=90) | (clientes["idade"] < 18)])) # verificando a qtd de clientes que tem idade incoerente

In [ ]:
idade_exclusao = clientes[(clientes["idade"] >=90) | (clientes["idade"] < 18)] #variavel das idades incoerentes

transacoes_exclusao = transacoes.merge(idade_exclusao, on="id_cliente")[['id_transacao', 'id_cliente', 'idade', 'valor_transacao']] #vinculando as planilhas para verificar possibilidade de exclusão dos clientes

transacoes_por_cliente = transacoes_exclusao.groupby("id_cliente")["id_transacao"].count() # quantidade de transações pelos clientes com idade incoerente

transacoes_por_cliente

- obs: acima verifiquei a quantidade de transações dos clientes que possuem idades incoerentes, no intuito de verificar se poderia excluir algum desses cliente da base de dados.

In [ ]:
transacoes_exclusao.isnull().sum() # poucas transações onde o valor é nulo

- obs: Como no grupo dos clientes com idade incoerente não tem muitos valores nulos, e a quantidade de transações é significativa, não é possivel fazer a exclusão. Opção 2: Trazendo o valor da mediana e da média para alterar essas idades incoerentes no dataframe.

In [ ]:
idade_valida = clientes[(clientes["idade"] <=90) | (clientes["idade"] >= 18)]['idade']

print(idade_valida.median())

print(idade_valida.mean())

mediana_valida = clientes[(clientes["idade"] <=90) | (clientes["idade"] >= 18)]['idade'].median()


In [ ]:
clientes_1 = clientes.copy() #fazendo uma copia para não modificar o dataframe

clientes_1.loc[(clientes_1["idade"] >= 90) | (clientes_1["idade"] < 18), "idade"] = mediana_valida #substituindo a idade pela mediana

clientes_1[(clientes_1["idade"] >=90) | (clientes_1["idade"] < 18)] #testando se tem clientes com idade incoerente
clientes_1[(clientes_1["idade"] <=90) | (clientes_1["idade"] > 18)]

- Criou-se um novo dataframe para não alterar o banco original. O novo banco possui clientes com idade maior igual a 18 anos e menor igual a 90 anos. Para os 25 valores encontrados que possuia idade incorente alterou-se a idade pela mediana da idade do banco de dados.

- <u> Verificando valores do score, possui valores nulos</u>

In [ ]:
registros_nulos = clientes_1[(clientes_1["score_credito"].isna())] # 180 registros

registros_nulos.groupby("segmento_cartao")["score_credito"].size() # qtd de registros por segmento

In [ ]:
registros_validos = clientes_1[(clientes_1["score_credito"].notna())] # 3820 registros com valor do score

estatisticas = registros_validos.groupby("segmento_cartao")[["score_credito","limite_credito"]].agg(["mean", "median"]) # calculando a media e mediana do score por cartão

estatisticas

- obs: Após o calculo, optei pela alteração do score nulo pela mediana por segmento.

In [ ]:
# transformando os valores nulos da coluna score_credito pela mediana
clientes_1.loc[clientes_1["score_credito"].isna(), "score_credito"] = (clientes_1.groupby("segmento_cartao")["score_credito"].transform("median"))

#clientes_1.info()

- <u>Conforme análise anterior, os valores de categoria (planilha transacoes) estão duplicados, logo é necessario unificar os tipos de categorias</u>

In [ ]:
# sorted(transacoes['categoria'].unique().tolist())
transacoes['categoria'] = transacoes['categoria'].apply(lambda x: str(x).title() if pd.notnull(x) else x) #padroniza o texto da coluna, cada valor ficará com iniciais maiúsculas, mas sem alterar valores nulos

sorted(transacoes['categoria'].unique().tolist()) #conferindo se possui 10 valores como o planejado

print(len(sorted(transacoes['categoria'].unique().tolist())))

- obs: Optou-se por modificar o dataframe já que a alteração não descaracteriza os valores, apenas padroniza.

- <u>A coluna valor_transacao possui valores negativos, e 2161 valores nulos. Qual o tratamento ideal para esse caso? ANALISAR</u>

In [ ]:
transacoes[(transacoes["valor_transacao"].isna())]

In [ ]:
valores_nega = transacoes[(transacoes["valor_transacao"] <= 0)] #definindo a variavel par os valores iguis e menores do que zero

print(len(valores_nega)) #quantidade de transações negativas

print(valores_nega)

In [ ]:
# Verificando se tem valores iguais que possam representar devoluções

#transacoes[(transacoes["id_cliente"] == 3251) & (transacoes["categoria"] == "Supermercado")]
#transacoes[(transacoes["id_cliente"] == 2599) & (transacoes["categoria"] == "Supermercado")]
#transacoes[(transacoes["id_cliente"] == 605) & (transacoes["categoria"] == "Supermercado")]
#transacoes[(transacoes["id_cliente"] == 1292) & (transacoes["categoria"] == "Educação")]
transacoes[(transacoes["id_cliente"] == 3371) & (transacoes["categoria"] == "Combustível")]

- acima eu tentei idenficar em alguns clientes se existiu transações positivas evidenciando posiiveis devoluções de valores.

obs: para os valores negativos transformei em valor positivo, por deduzir que seja um erro no dado. Veriquei alguns clientes que possuem valores negativos no intuito de encontrar valores "iguais" representando possíveis devoluções.

In [ ]:
transacoes_1 = transacoes.copy() # copiando o dataframe original para criar um novo

transacoes_1["valor_transacao"] = transacoes_1["valor_transacao"].abs()

transacoes_1[(transacoes_1["valor_transacao"] <= 0)]

valores nulos

In [ ]:
transacoes_1[(transacoes_1["valor_transacao"].isna())] 

porcentagem = transacoes_1["valor_transacao"].isnull().mean()*100 #qtos porcento da base representa os valores nulos?
porcentagem

- obs: Poderia excluir os valores nulos da base transacoes, pois representam 1% de todo o dataframe. Mas optei por verificar a mediana dos gastos, por depatarmento, pelos clientes que possuem esses valores nulos. 

In [ ]:
nulos = transacoes_1[transacoes_1["valor_transacao"].isna()]["id_cliente"].unique() #quem são os clientes que tem valores nulos

transacao_nulos = transacoes_1[transacoes_1["id_cliente"].isin(nulos)] # identificando as transações desses clientes

agrupado = transacao_nulos.groupby(['id_cliente', 'categoria'])['valor_transacao'].median() #mediana dos valores gastos por cliente e por categoria

agrupado

In [ ]:
agrupado.isna().sum()
#agrupado[agrupado.isna()]

In [ ]:
medianas = transacao_nulos.groupby(['id_cliente', 'categoria'])['valor_transacao'].transform('median') #variavel para transformação dos valores nulos

transacoes_1.loc[transacoes_1["id_cliente"].isin(nulos) & transacoes_1["valor_transacao"].isna(),"valor_transacao"] = medianas #modificando novamente o transacoes_1

transacoes_1[transacoes_1["valor_transacao"].isna()]["id_cliente"].unique()
print(len(transacoes_1[transacoes_1["valor_transacao"].isna()]["id_cliente"].unique()))

In [ ]:
# Tratando os valores nulos que ainda estão da base

transacoes_1["valor_transacao"] = transacoes_1["valor_transacao"].fillna(0)

transacoes_1.info()

- OBS: Transformei alguns valores nulos pelas medianas dos gastos por categoria de cada cliente, e por final os valores restantes transformei em nulos, já que são dados financeiros.

>**5º Estatística Descritiva (identificação e tratamento de outliers)**

- SLIDE 1 -> dados da base clientes. Trazer média de idade, faixa das idades; porcentagem de clientes por segmento; porcentagem de clientes por localidade; definir qual será o criterio para expor informação sobre limite de crédito; média de relacionamento, faixa de relacionamento; porcentagem de clientes pelo canal de aquisição do cartão.

In [ ]:
#clientes_1.info()

In [ ]:
# identificando valores em idade
print(clientes_1['idade'].mean())
print(clientes_1['idade'].max())
print(clientes_1['idade'].min())

In [ ]:
# totalidade por segmento
clientes_1['segmento_cartao'].value_counts()

clientes_1['segmento_cartao'].value_counts(normalize=True)*100

In [ ]:
# totalidade por região
clientes_1['regiao'].value_counts()
clientes_1['regiao'].value_counts(normalize=True)*100


In [ ]:
# limite por segmentação
clientes_1.groupby(['segmento_cartao'])['limite_credito'].mean().round(2)

clientes_1.groupby(['segmento_cartao'])['limite_credito'].median().round(2).sort_values(ascending=False)
 # utilizado o min pois tem valoresde limite muito altos nos segmentos
clientes_1.groupby(['segmento_cartao'])['limite_credito'].max().sort_values(ascending=False)

clientes_1.groupby(['segmento_cartao'])['limite_credito'].min().sort_values(ascending=False)

In [ ]:
# identificando valores em tempo_relacionamento_meses e transformando em anos
print(clientes_1['tempo_relacionamento_meses'].mean()/12)
print(clientes_1['tempo_relacionamento_meses'].median()/12)
print(clientes_1['tempo_relacionamento_meses'].max()/12)
print(clientes_1['tempo_relacionamento_meses'].min())

In [ ]:
# totalidade por canal de contratação
clientes_1['canal_aquisicao'].value_counts()
clientes_1['canal_aquisicao'].value_counts(normalize=True)*100

In [ ]:
#VERIFICANDO SE É INTERESSANTE COLOCAR UM GRAFICO DE PIZZA PARA ESSA ANALISE

# calcular percentuais
pct = clientes_1['canal_aquisicao'].value_counts(normalize=True) * 100
# plotar
plt.figure(figsize=(5,5))
plt.pie(
    pct,
    labels=pct.index,
    autopct='%1.1f%%',
    startangle=90
)

plt.title('Distribuição dos Canais de Aquisição (%)')
plt.show()

- SLIDE 2 -> dados da base transações. período de que corresponde a base de transações; média de quantidade de transações por cliente e média de gastos por cliente; quantidade e valores totais por local de consumo; porcentagem das transações pelo canal utilizado para as compras

In [ ]:
#transacoes_1.info()

In [ ]:
transacoes_1['valor_transacao'].sum() #total de gastos da carteira ativa

In [ ]:
transacoes_1.groupby(['id_cliente'])['id_transacao'].count() #qtd de transações por cliente
transacoes_1.groupby(['id_cliente'])['id_transacao'].count().mean() #média de transações
transacoes_1.groupby(['id_cliente'])['id_transacao'].count().median() #mediana
transacoes_1.groupby(['id_cliente'])['id_transacao'].count().describe() #informações estatiscas

- obs: os valores estão dispersos para trazer informações sobre a média (que está sendo elevada) de transações por clientes, utilizar analise de outliers

In [ ]:
qtd_transacoes = transacoes_1.groupby(['id_cliente'])['id_transacao'].count()
# calculos para verificação de outliers
q1 = qtd_transacoes.quantile(0.25) 
q3 = qtd_transacoes.quantile(0.75)
iqr = q3 - q1
print(q1)
print(q3)
print(iqr)

limite_superior = q3 + 1.5*iqr
print(limite_superior)

outliers = qtd_transacoes[qtd_transacoes > limite_superior]
outliers.shape

In [ ]:
qtd_transacoes_df = qtd_transacoes.reset_index()

sns.boxplot(y=qtd_transacoes_df['id_transacao'])
plt.title('Distribuição da Quantidade de Transações por Cliente')
plt.show()

In [ ]:
sem_outliers = qtd_transacoes_df[qtd_transacoes_df['id_transacao'] <= limite_superior]

print(qtd_transacoes_df['id_transacao'].mean())
print(sem_outliers['id_transacao'].mean())

- Após a analise com os outliers da quantidade de transações por cliente, decidi ficar com a média sem os outliers (51 transações), tendo em visto que fica mais próximo da mediana (54 transações).

In [ ]:
transacoes_1.groupby(['id_cliente'])['valor_transacao'].sum() #valor dos gastos por cliente
print(transacoes_1.groupby(['id_cliente'])['valor_transacao'].sum().mean()) #média de gastos
print(transacoes_1.groupby(['id_cliente'])['valor_transacao'].sum().median()) #mediana
transacoes_1.groupby(['id_cliente'])['valor_transacao'].sum().describe() #informações estatiscas

- obs: os valores estão dispersos para trazer informações sobre a média (que está sendo elevada) dos gastos por clientes, utilizar analise de outliers

In [ ]:
valor_gastos = transacoes_1.groupby(['id_cliente'])['valor_transacao'].sum()
# calculos para verificação de outliers
q1 = valor_gastos.quantile(0.25) 
q3 = valor_gastos.quantile(0.75)
iqr = q3 - q1
print(q1)
print(q3)
print(iqr)

lim_sup_gasto = q3 + 1.5*iqr
print(lim_sup_gasto)

out_gastos = valor_gastos[valor_gastos > lim_sup_gasto]
out_gastos.shape

In [ ]:
valor_gastos_df = valor_gastos.reset_index()

sns.boxplot(y=valor_gastos_df['valor_transacao'])
plt.title('Distribuição do Valor dos Gastos por Cliente')
plt.show()

In [ ]:
sem_out_gastos = valor_gastos_df[valor_gastos_df['valor_transacao'] <= lim_sup_gasto]
sem_out_gastos
print(valor_gastos_df['valor_transacao'].mean())
print(sem_out_gastos['valor_transacao'].mean())

- Como na analise anterior, decidi utilizar a metrica média sem os outliers (14247.32), tendo em visto ter poucos outliers em relação a quantidade de clientes, e ficar mais próximo da mediana (13034.57).

In [ ]:
#verificando periodo do banco de dados e se possui valores nulos
print(transacoes_1['data_transacao'].min())
print(transacoes_1['data_transacao'].max())
transacoes_1['data_transacao'].isna().value_counts()

In [ ]:
#utilizei a agregação de categoria pelo valor , transformando em uma tabela com os valores totais em ordem decresnte e a qtd de transações
categoria_df = transacoes_1.groupby('categoria')['valor_transacao'].agg(
    total_valor=lambda x: round(x.sum(), 2),
    qtd_transacoes='count'
).sort_values(by='qtd_transacoes', ascending=False)

categoria_df

para a apresentação dos valores por categoria, primeiro analisei o total dos gastos em ordem descrescente e depois pela quantidade de transações.

In [ ]:
transacoes_1['canal_transacao'].value_counts()
transacoes_1['canal_transacao'].value_counts(normalize=True)*100

verificando a porcentagem de transações por canal de utilização do consumo. para a apresentação analisar a possibilidade de inserir um grafico de pizza.

In [ ]:
canal_pct = transacoes_1['canal_transacao'].value_counts(normalize=True)*100
colors = ['#845ec2','#ffc755'] #definindo novas cores
# plotar
plt.figure(figsize=(4,4))
plt.pie(
    canal_pct,
    labels=canal_pct.index,
    autopct='%1.1f%%',
    colors=colors,
    startangle=90
)
plt.title('Distribuição dos Canais de Compras (%)')
plt.savefig('grafico_canais.png', dpi=300, bbox_inches='tight')
plt.show()

>**6º AGREGAÇÕES ENTRE AS PLANILHAS E VISUALIZAÇÃO COM A BIBLIOTECA SEABORN**

In [ ]:
#clientes_1.head()

- 1ª analise: planilha clientes SCORE X LIMITE , SCORE X SEGMENTO, REGIAO X SEGMENTO

In [ ]:
fig, ax = plt.subplots(figsize=(8,5)) 

sns.histplot(
    data=clientes_1, 
    x="score_credito", 
    hue="segmento_cartao", 
    bins=40, 
    kde=True, 
    alpha=0.5, 
    ax=ax 
)

ax.set_title("Score de Crédito por Segmento")
ax.set_xlabel("Score de Crédito") 
ax.set_ylabel("Segmento") 

plt.tight_layout() 
plt.show() 

clientes de maior score estão classificados nos segmentos Black e Platinum, enquanto o segmento Classic o score de credito conscentra-se entre 500 a 650 pontos de score

In [ ]:
fig, ax = plt.subplots(figsize=(8,5)) 

sns.histplot(
    data=clientes_1, 
    x="limite_credito", 
    hue="segmento_cartao", 
    bins=40, 
    kde=True, 
    alpha=0.5, 
    ax=ax 
)

ax.set_title("Limite de Crédito por Segmento")
ax.set_xlabel("Limite de Crédito") 
ax.set_ylabel("Segmento") 

plt.tight_layout() 
plt.show() 

quanto mais premium o cartão, maior o limite médio. A maior parte dos limites está abaixo de 50 mil, independentemente do segmento. Há limites que chegam a 400 mil, muito acima da média.

In [ ]:
clientes_1[['score_credito', 'limite_credito']].corr()

a correlação é proxima de zero, logo é muito baixa a correlação

Qual é a relação entre o score de crédito e o limite de crédito?

Há uma relação positiva, porém fraca. O score de crédito tem influência no limite, mas não é o principal determinante. A correlação de 0,14 indica que o score explica apenas uma pequena parte da variação do limite de crédito.

In [ ]:
sns.regplot(
    data=clientes_1,
    x='score_credito',
    y='limite_credito'
)

In [ ]:
clientes_1.groupby(['regiao', 'segmento_cartao']).size().unstack() #quantidade de clientes por segmento

In [ ]:
clientes_1.groupby(['regiao', 'segmento_cartao']).size().groupby(level=0).apply(lambda x: x / x.sum() * 100).unstack() #porcentagem dos clientes em cada segmento

In [ ]:
seg_reg = clientes_1.groupby(['regiao', 'segmento_cartao']).size().unstack()

tabela_pct = seg_reg.div(seg_reg.sum(axis=1), axis=0)

colors = ['#845ec2','#ffc752', '#f9f871', '#2c73d2' ]
tabela_pct.plot(kind='bar', stacked=True, figsize=(10,6), color=colors)

Os segmentos de cartão (Classic, Gold, Platinum, Black) se distribuem de forma diferente entre as regiões?

O gráfico mostra que a carteira por segmento não tem alterações relevantes entre as regiões. Não há concentração expressiva de segmentos em nenhuma região específica, indicando que o segmento do cartão não é fortemente influenciado pela localização.

- 2ª analise: entre planilhas CLIENTES X TRANSAÇÕES

In [ ]:
# criando um dataframe para explorar relações entre clientes_1 e transacoes_1

merge = transacoes_1.merge(clientes_1, on='id_cliente')
merge.head()

- Qual o segmento que utiliza mais o NEOCARD? Em qual categoria (já respondido anteriormente)?

In [ ]:
#qual segmento tem mais transações
merge.groupby('segmento_cartao')['id_transacao'].count().sort_values(ascending=False)

merge.groupby('segmento_cartao').agg(
    qtd_transacoes=('id_transacao', 'count'),
    total_valor=('valor_transacao',lambda x: round(x.sum(), 2))
).sort_values('qtd_transacoes', ascending=False)

In [ ]:
merge.groupby('segmento_cartao')['id_transacao'].count().pipe(
    lambda x: (x / x.sum()) * 100
).sort_values(ascending=False)

In [ ]:
grafico_seg = merge.groupby('segmento_cartao')['id_transacao'].count().sort_values(ascending=False)

grafico_seg.plot(kind='bar', figsize=(6,4))
plt.title('Transações por Segmento')
plt.ylabel('Valores')
plt.xlabel('Segmento')
plt.xticks(rotation=0)
plt.show()

In [ ]:
gold = merge[merge['segmento_cartao'] == "Gold"]
gold.groupby(['categoria'])['id_transacao'].count().pipe(
    lambda x: (x / x.sum()) * 100
).sort_values(ascending=False)

In [ ]:
grafico_gold = gold.groupby(['categoria'])['id_transacao'].count().pipe(
    lambda x: (x / x.sum()) * 100
).sort_values(ascending=False)

grafico_gold.plot(kind='bar', figsize=(13,4))
plt.title('Transações do Segmento GOLD')
plt.ylabel('% das Transações')
plt.xlabel('Estabelecimento')
plt.xticks(rotation=0)
plt.show()

In [ ]:
merge.groupby('segmento_cartao')['valor_transacao'].sum().pipe(
    lambda x: (x / x.sum()) * 100
).sort_values(ascending=False)

o segmento que mais utiliza é o GOLD, em supermercados, restaurantes e vestuario, convergindo com a analise pela totalidade de transações.


- Qual região utiliza mais? Como é o consumo dos clientes por região?

In [ ]:
merge.groupby('regiao')['id_transacao'].count().sort_values(ascending=False)

merge.groupby('regiao').agg(
    qtd_transacoes=('id_transacao', 'count'),
    total_valor=('valor_transacao',lambda x: round(x.sum(), 2))
).sort_values('qtd_transacoes', ascending=False)

a região SUDESTE utiliza mais e gasta mais

- Dentro de cada região, qual segmento usa mais o cartão? Em relação a região/segmento como seria analise de acordo com a porcentagem de utilização

In [ ]:
#Contagem por segmento e local
#merge.groupby(['regiao', 'segmento_cartao'])['id_transacao'].count()

In [ ]:
reg_seg = merge.groupby(['regiao', 'segmento_cartao']).agg(
    qtd_transacoes=('id_transacao', 'count'))

reg_seg['pct_transacoes'] = (reg_seg['qtd_transacoes'] / reg_seg['qtd_transacoes'].sum()) * 100

reg_seg.sort_values('qtd_transacoes', ascending=False)

trazendo a porcentagem de utilização do cartão por região e segmento. a região sudeste utiliza mais , ja que tem a carteira maior. ANALISAR de outra forma pois não caracteriza o comportamento

In [ ]:
# porcentagem dentro de cada região
reg_seg['pct_regiao'] = reg_seg.groupby(level=0)['qtd_transacoes'].transform(lambda x: x / x.sum() * 100)

reg_seg.sort_values(['regiao', 'pct_regiao'], ascending=[True, False])

In [ ]:
grafico_reg = reg_seg['pct_regiao'].unstack()

colors = ['#845ec2','#ffc752', '#f9f871', '#2c73d2' ]
grafico_reg.plot(kind='bar', figsize=(10,6), color=colors)

plt.title('Quantidade de Transações por Região e Segmento')
plt.ylabel('Qtd de Transações')
plt.xlabel('Região')
plt.xticks(rotation=0)
plt.legend(title='Segmento')
plt.show()

- OBS: visualmente vemos que a predominancia de utilização do cartão por região é do segmento CLASSIC, já que possui o maior de clientes, sendo apenas a na região Sudeste o segmento GOLD utiliza mais o cartão. 

Para evitar distorções causadas pela maior concentração de clientes no Sudeste, a análise foi feita considerando a participação percentual dos segmentos dentro de cada região. Observa-se que o segmento Classic é predominante em todas as regiões, mas sua participação varia conforme o perfil regional.

A distribuição dos segmentos é relativamente homogênea entre as regiões, sem diferenças marcantes. Isso indica que o perfil de segmento da carteira não varia de forma significativa por localização geográfica. 

In [ ]:
reg_cat = merge.groupby(['regiao', 'categoria']).agg(
    qtd_transacoes=('id_transacao', 'count'))

# porcentagem dentro de cada região
reg_cat['pct_regiao'] = reg_cat.groupby(level=0)['qtd_transacoes'].transform(lambda x: x / x.sum() * 100)

reg_cat.sort_values(['regiao', 'pct_regiao'], ascending=[True, False])

In [ ]:
grafico_consumo = reg_cat['pct_regiao'].unstack()

grafico_consumo.plot(kind='bar', stacked=True, figsize=(12,6), colormap='tab20')

plt.title('Distribuição Percentual das Categorias por Região')
plt.ylabel('Percentual (%)')
plt.xlabel('Região')
plt.xticks(rotation=0)
plt.legend(title='Categoria')
plt.show()

In [ ]:
sns.heatmap(grafico_consumo, annot=True, cmap='Blues')
plt.title('Heatmap de Percentual por Região e Categoria')
plt.show()

com o grafico de calor , fica mais clara que em todas as regiões o consumo é com supermercado seguido do restaurante.

Em todas as regiões, a categoria Supermercado aparece como a mais representativa, indicando que os clientes utilizando o cartÃO em compras essenciais. Em seguida, categorias como Restaurante, Farmácia e Combustível mantêm participação relevante, reforçando o perfil de uso cotidiano do NEOCARD.

Não há regiões com concentração atípica em categorias específicas, o que sugere que o padrão de consumo dos clientes é semelhante independentemente da localização.


- como é o comportamento em valores?

In [ ]:
vlr_reg_gasto = merge.groupby(['regiao', 'categoria']).agg(
    vlr_transacoes=('valor_transacao', 'sum'))

# porcentagem dentro de cada região
vlr_reg_gasto['pct_regiao'] = vlr_reg_gasto.groupby(level=0)['vlr_transacoes'].transform(lambda x: x / x.sum() * 100)

vlr_reg_gasto.sort_values(['regiao', 'pct_regiao'], ascending=[True, False])

In [ ]:
grafico_gasto = vlr_reg_gasto['pct_regiao'].unstack()

sns.heatmap(grafico_gasto, annot=True, cmap='Greens')
plt.title('Heatmap de Percentual por Região e Categoria')
plt.show()

In [ ]:
cat_corr = merge.groupby('categoria').agg(
    qtd_transacoes=('id_transacao', 'count'),
    valor_total=('valor_transacao', 'sum')
)

cat_corr.corr()

Categorias com mais transações não necessariamente são as que geram mais receita. O volume de uso não explica bem o valor financeiro movimentado. Existem categorias com poucas transações, mas que movimentam muito dinheiro (ex.: Eletrônicos, Viagem). Existem categorias com muitas transações, mas ticket baixo (ex.: Supermercado).

A análise percentual por região mostra que as categorias Viagem, Eletrônicos e Supermercado são as mais representativas em todas as regiões, concentrando entre 20% e 23% do valor total de gastos. Esse padrão indica que o uso do NEOCARD está fortemente associado a compras essenciais (Supermercado), itens de maior valor agregado (Eletrônicos) e despesas de viagem.

>**7º CONCLUSÕES EXECUTIVAS - RECOMENDAÇÃOES**

- Qual categoria alavancar em numero de transações?

In [ ]:
cat_corr = merge.groupby('categoria').agg(
    qtd_transacoes=('id_transacao', 'count'),
    valor_total=('valor_transacao',lambda x: round(x.sum(), 2)))

cat_corr['ticket_medio'] = cat_corr['valor_total'] / cat_corr['qtd_transacoes']

cat_corr.sort_values('ticket_medio', ascending=False)

In [ ]:
cat_corr[['ticket_medio', 'qtd_transacoes']].corr()

A analise da correlação da quantidade de transações pelo ticket médio, tem o indice -0,27, que determina que a correlação e baixa e negativa. Tickect médio alto menos transações.

**1. Intensificar campanhas e incentivos nas categorias de Eletrônicos e Viagem para aumentar a frequência de uso, já que essas categorias apresentam ticket médio elevado e, portanto, maior potencial de geração de receita por transação.**

- Estratégias amplas para o segmento CLASSSIC e GOLD

Segundo analises anteriores os segmentos Classic e Gold utilizam mais o cartão NeoCard, mesmo que em categorias com tickect médio, medidas de fortalecimento são essenciais para fidelizar esses clientes.

**2. Promover Estratégias comerciais amplas, focadas ao segmento Classic e Gold, grande publico que utiliza o cartão em compras essenciais.**

- Panorama para possibilitar upgrades conforme tempo e score

In [ ]:
relacionamento = merge.groupby('segmento_cartao').agg(
    tempo_medio_relacionamento=('tempo_relacionamento_meses', 'mean'),
    score_medio=('score_credito', 'mean'),
    qtd_clientes=('id_cliente', 'nunique'))

relacionamento

In [ ]:
sns.heatmap(relacionamento[['tempo_medio_relacionamento', 'score_medio']].corr(), annot=True)


correlação baixa

In [ ]:
upgrade_por_segmento = merge[merge['score_credito'] >= 700].groupby('segmento_cartao').agg(
    qtd_clientes=('id_cliente', 'nunique'),
    score_medio=('score_credito', 'mean'),
    tempo_medio=('tempo_relacionamento_meses', 'mean')
)
upgrade_por_segmento

Filtrei os clientes com score igual superior a score 700, vito que em analise visual anterior, não há caudas longas, não há distorção, não há valores extremos que puxem a média.

**3. Trazer benefícios a clientes que possuem risco baixo.**

Conclusão:  
O segmento Black concentra clientes maduros, de baixo risco e com forte engajamento. Tem potencial para upgrade de benefícios e aumento de limite.

Platinum é o segundo segmento com maior potencial de upgrade. A combinação de score elevado e tempo longo indica clientes confiáveis e com espaço para expansão de produtos.

Gold apresenta potencial moderado. Pode receber upgrade gradual, campanhas de ativação e benefícios intermediários.

Classic não apresenta perfil ideal para upgrade. Score mais baixo e base reduzida indicam que o foco deve ser manutenção e engajamento, não expansão.

Analise de linha de tempo

In [ ]:
#coluna mes
merge['mes'] = merge['data_transacao'].dt.to_period('M')

merge.head()

In [ ]:
#agrupando informações por mes

consumo_tempo = merge.groupby('mes').agg(
    qtd_transacoes=('id_transacao', 'count'),
    valor_total=('valor_transacao', 'sum'),
    ticket_medio=('valor_transacao', 'mean')
)
consumo_tempo

Achei desnecessario mencionar ou criar promoções em determinados meses, pensando em alavancar a atulização, visto que os valores, quantidades de transações e ticket medio são muito parecidos ao longo do ano.